In [ ]:
#@title 0. Instalar dependencias y montar Drive
from google.colab import drive
drive.mount('/content/drive')

# Instalar detectron2 desde fuente
import sys, os, distutils.core

# Detectron2 (fast local install)
!git clone -q 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install -q {' '.join([f"'{x}'" for x in dist.install_requires])}
import sys, os
sys.path.insert(0, os.path.abspath('./detectron2'))

# Otros paquetes
!pip install -q rasterio pyproj fiona matplotlib albumentations tqdm pycocotools pandas scikit-image

In [2]:
#@title 1. Rutas, constantes y semilla

import os, json, random
import numpy as np
from pathlib import Path

# Base del proyecto
BASE = Path("/content/drive/MyDrive/juniper_mapper/JuniperMapper")

# COCO instancia (para métricas IoU/S-IoU por instancia)
PI_COCO_JSON = str(BASE/"Photo_Interpretation_Data/Test/Annotations/Test.json")
PI_IMG_DIR   = str(BASE/"Photo_Interpretation_Data/Test/Images")

FW_COCO_JSON = str(BASE/"Field_Work_Data/External_Val_Data/Annotations/FW.json")
FW_IMG_DIR   = str(BASE/"Field_Work_Data/External_Val_Data/Images")

# Máscaras GT semánticas (binarias 0/1) para métricas pixelares/cobertura
GT_PI = str(BASE/"Photo_Interpretation_Data/Test/Annotations/Masks")
GT_FW = str(BASE/"Field_Work_Data/External_Val_Data/Annotations/Masks")

# Salidas Mask R-CNN
OUT_PI  = str(BASE/"MaskRCNN/output_PI")
OUT_FW  = str(BASE/"MaskRCNN/output_FW")
PRED_PI_TH = f"{OUT_PI}/Predictions_theta_star"  # binarias a θ*
PRED_FW_TH = f"{OUT_FW}/Predictions_theta_star"

for d in [OUT_PI, OUT_FW, PRED_PI_TH, PRED_FW_TH, f"{OUT_PI}/csv", f"{OUT_PI}/tables", f"{OUT_PI}/plots"]:
    os.makedirs(d, exist_ok=True)

# GSD (fallback en m/px si no hay CRS proyectado en GeoTIFF)
GSD_FALLBACK = 0.13

# Bins de tamaño (m²)
SIZE_BINS = [
    ("XS",  0.13,  1.72),
    ("S",   1.72,  3.62),
    ("M",   3.62,  9.08),
    ("L",   9.08, 20.82),
    ("XL", 20.82, 41.06),
    ("XXL",41.06, float("inf")),
]
def size_label(area_m2: float):
    for name, lo, hi in SIZE_BINS:
        if lo <= area_m2 < hi:
            return name
    return "XS"

# Semilla
SEED = 1337
random.seed(SEED); np.random.seed(SEED)

In [ ]:
#@title 2. Registro de datasets (COCO instancia + metadatos)

from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog

register_coco_instances("pi_test_coco", {}, PI_COCO_JSON, PI_IMG_DIR)
register_coco_instances("fw_test_coco", {}, FW_COCO_JSON, FW_IMG_DIR)

print("Datasets registrados:", list(MetadataCatalog.keys()))

In [ ]:
#@title 3. Configuración Mask R-CNN (R101-C4) y predictor

import torch
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
import warnings

# Pesos
CKPT_PATH = str(BASE / "model/Mask_RCNN_ResNet101-C4.pth")

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_101_C4_3x.yaml"))
cfg.MODEL.WEIGHTS = CKPT_PATH
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.05  # mínimo, barremos manualmente
cfg.TEST.DETECTIONS_PER_IMAGE = 256
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

predictor = DefaultPredictor(cfg)
print("Predictor listo | Device:", cfg.MODEL.DEVICE)

# Silenciar el warning de meshgrid
warnings.filterwarnings(
    "ignore",
    message="torch.meshgrid:.*indexing argument",
    category=UserWarning
)

In [5]:
#@title 4. Predictor (sin TTA) y fusión por IoU rápido

import cv2

_RAW_CACHE = {}
def _key(img_path, view):
    return f"{img_path}::{view}"

@torch.no_grad()
def _predict_raw(img_bgr, predictor, cache_key=None):
    if cache_key and cache_key in _RAW_CACHE:
        return _RAW_CACHE[cache_key]
    out = predictor(img_bgr)
    inst = out["instances"].to("cpu")
    masks  = inst.pred_masks.numpy().astype(bool) if inst.has("pred_masks") else np.zeros((0, *img_bgr.shape[:2]), bool)
    scores = inst.scores.numpy() if inst.has("scores") else np.zeros((0,), float)
    if cache_key:
        _RAW_CACHE[cache_key] = (masks, scores)
    return masks, scores

# Compatibilidad con funciones que esperaban (img_path, view=...)
def _predict_raw_view(img_bgr, predictor, img_path, view="orig"):
    return _predict_raw(img_bgr, predictor, cache_key=_key(img_path, view))

def _bbox_from_mask(m):
    ys = np.any(m, axis=1)
    xs = np.any(m, axis=0)
    if not ys.any() or not xs.any():
        return (0, 0, 0, 0)
    y = np.where(ys)[0]
    x = np.where(xs)[0]
    return (int(y[0]), int(x[0]), int(y[-1]), int(x[-1]))

def _box_iou(b1, b2):
    y1 = max(b1[0], b2[0])
    x1 = max(b1[1], b2[1])
    y2 = min(b1[2], b2[2])
    x2 = min(b1[3], b2[3])
    inter = max(0, y2 - y1 + 1) * max(0, x2 - x1 + 1)
    a1 = (b1[2] - b1[0] + 1) * (b1[3] - b1[1] + 1)
    a2 = (b2[2] - b2[0] + 1) * (b2[3] - b2[1] + 1)
    uni = a1 + a2 - inter
    return inter / uni if uni > 0 else 0.0

def _merge_by_iou_fast(masks_sorted, scores_sorted, iou_thr=0.85, downsample=2):
    N = masks_sorted.shape[0]
    if N <= 1:
        return masks_sorted, scores_sorted
    small = masks_sorted[:, ::downsample, ::downsample] if downsample > 1 else masks_sorted
    boxes = [_bbox_from_mask(m) for m in masks_sorted]
    kept_idx, kept_small, kept_boxes = [], [], []
    for i in range(N):
        keep = True
        for j in range(len(kept_idx)):
            if _box_iou(boxes[i], kept_boxes[j]) < iou_thr:
                continue
            inter = np.logical_and(small[i], kept_small[j]).sum()
            union = np.logical_or(small[i], kept_small[j]).sum()
            if union > 0 and inter / union >= iou_thr:
                keep = False
                break
        if keep:
            kept_idx.append(i)
            kept_small.append(small[i])
            kept_boxes.append(boxes[i])
    kept_idx = np.asarray(kept_idx, int)
    return masks_sorted[kept_idx], scores_sorted[kept_idx]

def predict_instances(img_path, predictor, score_thr=0.5, iou_merge=0.85, topk=256):
    img = cv2.imread(img_path)
    if img is None:
        return [], []
    H, W = img.shape[:2]

    # Predicción única sin TTA
    m0, s0 = _predict_raw(img, predictor, cache_key=_key(img_path, "orig"))
    keep = s0 >= float(score_thr)
    m0, s0 = m0[keep], s0[keep]

    masks_list = [m0]
    scores_list = [s0]

    # Concatenación segura (permitir que todo sea vacío)
    masks_parts  = [m for m in masks_list  if isinstance(m, np.ndarray) and m.size  > 0]
    scores_parts = [s for s in scores_list if isinstance(s, np.ndarray) and s.size > 0]
    if len(masks_parts) == 0 or len(scores_parts) == 0:
        return [], []

    masks  = np.concatenate(masks_parts,  axis=0)
    scores = np.concatenate(scores_parts, axis=0)
    if masks.shape[0] == 0:
        return [], []

    order = np.argsort(-scores)[:topk]
    masks, scores = masks[order], scores[order]
    masks, scores = _merge_by_iou_fast(masks, scores, iou_thr=iou_merge, downsample=2)
    return [m for m in masks], [float(s) for s in scores]

In [6]:
#@title 5. Utilidades: GSD, GT, métricas, pixelares

import rasterio
from pycocotools.coco import COCO
from scipy.optimize import linear_sum_assignment
from contextlib import redirect_stdout, redirect_stderr
import io

def load_coco_silent(json_path):
    buf = io.StringIO()
    with redirect_stdout(buf), redirect_stderr(buf):
        coco = COCO(json_path)
    return coco

def image_gsd_m(geotiff_path, fallback=GSD_FALLBACK):
    try:
        with rasterio.open(geotiff_path) as src:
            tr = src.transform
            if src.crs and src.crs.is_projected:
                px_m = abs(tr.a); py_m = abs(tr.e)
            else:
                return float(fallback)
            if px_m>0 and py_m>0: return float((px_m+py_m)/2.0)
    except Exception:
        pass
    return float(fallback)

def ann_masks_for_img(coco: COCO, img_id: int, H: int, W: int):
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)
    masks = []
    for a in anns:
        m = coco.annToMask(a).astype(bool)
        if m.shape != (H, W):
            m = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
        masks.append(m)
    return masks

def iou_matrix(pred_masks, gt_masks):
    if len(pred_masks)==0 or len(gt_masks)==0:
        return np.zeros((len(pred_masks), len(gt_masks)), float)
    P = np.array([p.astype(bool) for p in pred_masks])
    G = np.array([g.astype(bool) for g in gt_masks])
    inter = (P[:,None] & G[None,:]).sum(axis=(2,3))
    union = (P[:,None] | G[None,:]).sum(axis=(2,3))
    return np.divide(inter, union, out=np.zeros_like(inter, dtype=float), where=union>0)

def eval_iou_at_threshold_hungarian(pred_masks, gt_masks, iou_thr=0.5):
    iou = iou_matrix(pred_masks, gt_masks)
    if iou.size==0: return dict(tp=0,fp=len(pred_masks),fn=len(gt_masks))
    cost = 1.0 - iou
    ri, cj = linear_sum_assignment(cost)
    matched_pred=set(); matched_gt=set()
    for i,j in zip(ri,cj):
        if iou[i,j] >= iou_thr:
            matched_pred.add(i); matched_gt.add(j)
    tp = len(matched_pred); fp = len(pred_masks)-tp; fn = len(gt_masks)-tp
    P = tp/(tp+fp) if (tp+fp)>0 else 0.0
    R = tp/(tp+fn) if (tp+fn)>0 else 0.0
    F1= 2*P*R/(P+R) if (P+R)>0 else 0.0
    return dict(tp=tp,fp=fp,fn=fn,precision=P,recall=R,f1=F1)

def siou_pred(p_mask, gt_masks):
    matches = [g for g in gt_masks if np.any(p_mask & g)]
    if not matches: return 0.0
    union_gt = np.any(np.stack(matches,0),0)
    inter = np.logical_and(p_mask, union_gt).sum()
    den = union_gt.sum()
    return float(inter/den) if den>0 else 0.0

def siou_label(l_mask, pred_masks):
    matches = [p for p in pred_masks if np.any(l_mask & p)]
    if not matches: return 0.0
    union_pr = np.any(np.stack(matches,0),0)
    inter = np.logical_and(l_mask, union_pr).sum()
    den = l_mask.sum()
    return float(inter/den) if den>0 else 0.0

def eval_siou_at_threshold(pred_masks, pred_scores, gt_masks, siou_thr=0.5):
    order = np.argsort(-np.asarray(pred_scores))
    tp=fp=0
    for i in order:
        s = siou_pred(pred_masks[i], gt_masks)
        if s>=siou_thr: tp+=1
        else: fp+=1
    fn=0
    for g in gt_masks:
        if siou_label(g, pred_masks) < siou_thr: fn+=1
    P=tp/(tp+fp) if (tp+fp)>0 else 0.0
    R=tp/(tp+fn) if (tp+fn)>0 else 0.0
    F1=2*P*R/(P+R) if (P+R)>0 else 0.0
    return dict(tp=tp,fp=fp,fn=fn,precision=P,recall=R,f1=F1)

def pixel_metrics(pred_binary: np.ndarray, gt_binary: np.ndarray):
    y_pred = pred_binary.astype(np.uint8).ravel()
    y_true = gt_binary.astype(np.uint8).ravel()
    tp = int(np.sum((y_true==1)&(y_pred==1)))
    tn = int(np.sum((y_true==0)&(y_pred==0)))
    fp = int(np.sum((y_true==0)&(y_pred==1)))
    fn = int(np.sum((y_true==1)&(y_pred==0)))
    total = tp+tn+fp+fn
    acc = (tp+tn)/total if total>0 else 0.0
    iou_fg = tp/(tp+fp+fn) if (tp+fp+fn)>0 else 0.0
    iou_bg = tn/(tn+fp+fn) if (tn+fp+fn)>0 else 0.0
    miou = 0.5*(iou_fg+iou_bg)
    w0 = (tn+fp)/total if total>0 else 0.0
    w1 = (tp+fn)/total if total>0 else 0.0
    fwiou = w0*iou_bg + w1*iou_fg
    return dict(pACC=acc, mIoU=miou, fwIoU=fwiou)

In [7]:
#@title 6. Barrido de θ_score (criterio S-IoU@0.5 y 4 curvas) + guardados

import pandas as pd, matplotlib.pyplot as plt
from tqdm.auto import tqdm

def evaluate_split_at_score(coco, ims, img_dir, predictor, score, write_bin_dir=None, tag_out=None):
    """
    Evalúa un split a un umbral de score dado. Calcula:
      - F1/Prec/Rec para IoU@{0.5,0.75} con asignación Húngara
      - F1/Prec/Rec para S-IoU@{0.5,0.75}
      - Métricas pixelares medias (pACC, mIoU, fwIoU)
    Además, si write_bin_dir y tag_out están definidos, guarda las máscaras binarizadas predichas.
    Es robusta si una imagen no carga (cv2.imread devuelve None) o si no hay detecciones.
    """
    res_counts = {("IoU", t): dict(tp=0, fp=0, fn=0) for t in (0.5, 0.75)}
    res_counts.update({("S-IoU", t): dict(tp=0, fp=0, fn=0) for t in (0.5, 0.75)})
    pix_agg = dict(pACC=[], mIoU=[], fwIoU=[])

    for im in ims:
        fpath = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fpath)
        if img is None:
            # Ruta corrupta o no legible → saltar sin romper el flujo
            continue
        H, W = img.shape[:2]

        # GT y predicciones
        gt_masks = ann_masks_for_img(coco, im["id"], H, W)
        pred_masks, pred_scores = predict_instances(fpath, predictor, score_thr=score)

        # Pixelares (promediadas al final)
        pred_bin = np.any(np.stack(pred_masks, 0), 0) if len(pred_masks) > 0 else np.zeros((H, W), bool)
        gt_bin   = np.any(np.stack(gt_masks, 0), 0)   if len(gt_masks)   > 0 else np.zeros((H, W), bool)
        pm = pixel_metrics(pred_bin, gt_bin)
        for k, v in pm.items():
            pix_agg[k].append(v)

        # Globales por umbral
        for t in (0.5, 0.75):
            m_iou = eval_iou_at_threshold_hungarian(pred_masks, gt_masks, iou_thr=t)
            for k in ("tp", "fp", "fn"):
                res_counts[("IoU", t)][k] += m_iou[k]

            m_siou = eval_siou_at_threshold(pred_masks, pred_scores, gt_masks, siou_thr=t)
            for k in ("tp", "fp", "fn"):
                res_counts[("S-IoU", t)][k] += m_siou[k]

        # Guardado opcional de binarios
        if write_bin_dir and tag_out:
            try:
                with rasterio.open(fpath) as src:
                    meta = src.meta.copy()
                    meta.update(count=1, dtype=rasterio.uint8, nodata=0)
                os.makedirs(write_bin_dir, exist_ok=True)
                out_path = os.path.join(write_bin_dir, os.path.basename(fpath).replace("Img_", "Mask_"))
                with rasterio.open(out_path, "w", **meta) as dst:
                    dst.write(pred_bin.astype(np.uint8), 1)
            except Exception:
                # Si el GeoTIFF no se puede abrir/escribir, continuar sin interrumpir
                pass

    def _agg_to_metrics(d):
        tp, fp, fn = d["tp"], d["fp"], d["fn"]
        P = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        R = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
        return dict(tp=tp, fp=fp, fn=fn, precision=P, recall=R, f1=F1)

    metrics = {k: _agg_to_metrics(v) for k, v in res_counts.items()}
    pix_summary = {k: float(np.mean(v)) if len(v) > 0 else 0.0 for k, v in pix_agg.items()}
    return dict(metrics=metrics, pixel_metrics=pix_summary)

def _plot_four_curves(df, split_name, theta_star=None, out_dir=OUT_PI):
    plt.rcParams.update({
        "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
        "legend.fontsize": 10, "figure.dpi": 150
    })
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.plot(df["theta"], df["F1_IoU_0p5"],   marker="o",  label="IoU @ 0.5")
    ax.plot(df["theta"], df["F1_IoU_0p75"],  marker="s",  label="IoU @ 0.75")
    ax.plot(df["theta"], df["F1_SIoU_0p5"],  marker="^",  label="S-IoU @ 0.5")
    ax.plot(df["theta"], df["F1_SIoU_0p75"], marker="D",  label="S-IoU @ 0.75")

    if theta_star is not None:
        ax.axvline(float(theta_star), linestyle="--", linewidth=1)

    ax.set_xlabel("θ_score")
    ax.set_ylabel("F1-score (%)")
    ax.set_title(f"{split_name} — F1 vs θ_score (IoU y S-IoU) — Mask R-CNN")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", frameon=False)

    os.makedirs(f"{out_dir}/plots", exist_ok=True)
    out_path = os.path.join(out_dir, "plots", f"F1_vs_theta_{split_name.replace(' ','_')}_4curves.png")
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()
    return out_path

def sweep_theta(coco_json, img_dir, predictor, scores=None, title="Split", out_dir=OUT_PI):
    if scores is None: scores = np.linspace(0.05, 0.95, 19)

    coco = load_coco_silent(coco_json)
    ims = coco.loadImgs(coco.getImgIds())

    rows=[]
    for s in tqdm(scores, desc=f"Barrido θ_score — {title}", ncols=0):
        res = evaluate_split_at_score(coco, ims, img_dir, predictor, s)
        row = {
            "theta": float(s),
            "F1_IoU_0p5":   100*res["metrics"][("IoU",  0.5)]["f1"],
            "F1_IoU_0p75":  100*res["metrics"][("IoU",  0.75)]["f1"],
            "F1_SIoU_0p5":  100*res["metrics"][("S-IoU",0.5)]["f1"],
            "F1_SIoU_0p75": 100*res["metrics"][("S-IoU",0.75)]["f1"],
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    best = df.iloc[df["F1_SIoU_0p5"].idxmax()]
    best_theta = float(best["theta"])

    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"curves_F1_4lines_{title}.csv")
    df.to_csv(csv_path, index=False)
    png_path = _plot_four_curves(df, split_name=title, theta_star=best_theta, out_dir=out_dir)

    print(f"{title}: θ* (S-IoU@0.5) = {best_theta:.2f} | F1={best['F1_SIoU_0p5']:.2f}%")
    return best_theta, csv_path, png_path

In [ ]:
#@title 7. Ejecutar calibración θ* y evaluación final (global/size/pixel) + guardados

from IPython.display import display

coco_pi = load_coco_silent(PI_COCO_JSON)
ims_pi  = coco_pi.loadImgs(coco_pi.getImgIds())
coco_fw = load_coco_silent(FW_COCO_JSON)
ims_fw  = coco_fw.loadImgs(coco_fw.getImgIds())

theta_pi, csv_pi, png_pi = sweep_theta(PI_COCO_JSON, PI_IMG_DIR, predictor, title="PI", out_dir=OUT_PI)
theta_fw, csv_fw, png_fw = sweep_theta(FW_COCO_JSON, FW_IMG_DIR, predictor, title="FW", out_dir=OUT_PI)

def evaluate_split_tables(coco, ims, img_dir, predictor, theta, out_bin_dir):
    res = evaluate_split_at_score(coco, ims, img_dir, predictor, theta, write_bin_dir=out_bin_dir, tag_out=True)
    rows = []
    for (metric, thr), d in res["metrics"].items():
        rows.append(dict(
            Metric=metric, Thr=thr,
            TP=d["tp"], FP=d["fp"], FN=d["fn"],
            Precision=100*d["precision"], Recall=100*d["recall"], F1_score=100*d["f1"]
        ))
    df_global = pd.DataFrame(rows).sort_values(["Metric","Thr"]).reset_index(drop=True)
    return res, df_global

print("\nPI (θ*):")
res_pi_star, df_pi_global = evaluate_split_tables(coco_pi, ims_pi, PI_IMG_DIR, predictor, theta_pi, PRED_PI_TH)

print("\nFW (θ*):")
res_fw_star, df_fw_global = evaluate_split_tables(coco_fw, ims_fw, FW_IMG_DIR, predictor, theta_fw, PRED_FW_TH)

def sizewise_metrics(coco, ims, img_dir, predictor, theta):
    """
    Calcula métricas por bin de tamaño a IoU@0.5 y S-IoU@0.5.
    Devuelve un DataFrame con columnas:
      Size, IoU_P, IoU_R, IoU_F1, S-IoU_P, S-IoU_R, S-IoU_F1
    Es robusta si una imagen no carga (cv2.imread devuelve None) o si no hay detecciones.
    """
    sizewise = {name: {("IoU", 0.5): dict(tp=0, fp=0, fn=0),
                       ("S-IoU", 0.5): dict(tp=0, fp=0, fn=0)} for name, _, _ in SIZE_BINS}
    sizewise["All"] = {("IoU", 0.5): dict(tp=0, fp=0, fn=0),
                       ("S-IoU", 0.5): dict(tp=0, fp=0, fn=0)}

    for im in ims:
        fpath = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fpath)
        if img is None:
            # Imagen ilegible → saltar
            continue
        H, W = img.shape[:2]
        gsd = image_gsd_m(fpath, fallback=GSD_FALLBACK)

        gt_masks = ann_masks_for_img(coco, im["id"], H, W)
        gt_areas = [float(m.sum()) * (gsd ** 2) for m in gt_masks]
        pred_masks, pred_scores = predict_instances(fpath, predictor, score_thr=theta)

        # IoU @ 0.5: asignación greedy por máximo IoU (una etiqueta por GT)
        iou_mat = iou_matrix(pred_masks, gt_masks) if (len(pred_masks) > 0 and len(gt_masks) > 0) else np.zeros((len(pred_masks), len(gt_masks)))
        assigned = set()
        if iou_mat.size > 0:
            order = np.argsort(-iou_mat.max(axis=1))
            for i in order:
                j = int(np.argmax(iou_mat[i]))
                if iou_mat[i, j] >= 0.5 and j not in assigned:
                    assigned.add(j)
                    sname = size_label(gt_areas[j])
                    sizewise[sname][("IoU", 0.5)]["tp"] += 1
                    sizewise["All"][("IoU", 0.5)]["tp"] += 1

        # FNs: GT no asignadas
        for j in range(len(gt_masks)):
            if j not in assigned:
                sname = size_label(gt_areas[j])
                sizewise[sname][("IoU", 0.5)]["fn"] += 1
                sizewise["All"][("IoU", 0.5)]["fn"] += 1

        # FPs: predicciones sin GT con IoU>=0.5
        for i, p in enumerate(pred_masks):
            if iou_mat.shape[1] == 0 or iou_mat[i].max() < 0.5:
                a_m2 = float(p.sum()) * (gsd ** 2)
                sname = size_label(a_m2)
                sizewise[sname][("IoU", 0.5)]["fp"] += 1
                sizewise["All"][("IoU", 0.5)]["fp"] += 1

        # S-IoU @ 0.5: por etiqueta y por predicción
        for j, g in enumerate(gt_masks):
            s = siou_label(g, pred_masks)
            sname = size_label(gt_areas[j])
            if s >= 0.5:
                sizewise[sname][("S-IoU", 0.5)]["tp"] += 1
                sizewise["All"][("S-IoU", 0.5)]["tp"] += 1
            else:
                sizewise[sname][("S-IoU", 0.5)]["fn"] += 1
                sizewise["All"][("S-IoU", 0.5)]["fn"] += 1

        for p in pred_masks:
            if siou_pred(p, gt_masks) < 0.5:
                a_m2 = float(p.sum()) * (gsd ** 2)
                sname = size_label(a_m2)
                sizewise[sname][("S-IoU", 0.5)]["fp"] += 1
                sizewise["All"][("S-IoU", 0.5)]["fp"] += 1

    def _prf(d):
        tp, fp, fn = d["tp"], d["fp"], d["fn"]
        P = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        R = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
        return dict(precision=P, recall=R, f1=F1)

    rows = []
    for s in [b[0] for b in SIZE_BINS] + ["All"]:
        ri = _prf(sizewise[s][("IoU", 0.5)])
        rs = _prf(sizewise[s][("S-IoU", 0.5)])
        rows.append({
            "Size": s,
            "IoU_P": 100 * ri["precision"], "IoU_R": 100 * ri["recall"], "IoU_F1": 100 * ri["f1"],
            "S-IoU_P": 100 * rs["precision"], "S-IoU_R": 100 * rs["recall"], "S-IoU_F1": 100 * rs["f1"]
        })

    return pd.DataFrame(rows)

df_pi_sz = sizewise_metrics(coco_pi, ims_pi, PI_IMG_DIR, predictor, theta_pi)
df_fw_sz = sizewise_metrics(coco_fw, ims_fw, FW_IMG_DIR, predictor, theta_fw)

df_pix = pd.DataFrame([
    dict(Split="PI-test", **res_pi_star["pixel_metrics"]),
    dict(Split="FW-test", **res_fw_star["pixel_metrics"]),
])

df_pi_global.insert(0, "Data", f"PI-test (θ={theta_pi:.2f})")
df_fw_global.insert(0, "Data", f"FW-test (θ={theta_fw:.2f})")
df_all = pd.concat([df_pi_global, df_fw_global], ignore_index=True)

display(df_all); display(df_pi_sz); display(df_fw_sz); display(df_pix)

df_all.to_csv(f"{OUT_PI}/csv/global_metrics_MaskRCNN_theta_star.csv", index=False)
with open(f"{OUT_PI}/tables/table_global_MaskRCNN_theta_star.tex", "w") as f:
    f.write(df_all.to_latex(index=False, float_format="%.4f"))

df_pi_sz.to_csv(f"{OUT_PI}/csv/sizewise_PI_MaskRCNN_theta_star.csv", index=False)
df_fw_sz.to_csv(f"{OUT_PI}/csv/sizewise_FW_MaskRCNN_theta_star.csv", index=False)
with open(f"{OUT_PI}/tables/table_sizewise_PI_MaskRCNN_theta_star.tex","w") as f:
    f.write(df_pi_sz.to_latex(index=False, float_format="%.2f"))
with open(f"{OUT_PI}/tables/table_sizewise_FW_MaskRCNN_theta_star.tex","w") as f:
    f.write(df_fw_sz.to_latex(index=False, float_format="%.2f"))

df_pix.to_csv(f"{OUT_PI}/csv/pixel_metrics_MaskRCNN_theta_star.csv", index=False)
with open(f"{OUT_PI}/tables/table_pixel_metrics_MaskRCNN_theta_star.tex","w") as f:
    f.write(df_pix.to_latex(index=False, float_format="%.4f"))

print("\nListo: global/size/pixel guardados y curvas en:", csv_pi, "—", csv_fw)

In [ ]:
#@title 8. Cobertura y Densidad (θ*) — WS baseline y WS calibrado (FW)

import math
from scipy import ndimage
from sklearn.metrics import mean_squared_error, r2_score

def read_mask(path):
    with rasterio.open(path) as src:
        return src.read(1)

def raster_valid_area_ha(img_path, valid_mask=None):
    with rasterio.open(img_path) as src:
        H,W = src.height, src.width
        tr, crs, bounds = src.transform, src.crs, src.bounds
        nodata = src.nodata
        if valid_mask is None:
            try:
                arr = src.read(1)
                valid_mask = (arr != nodata) if nodata is not None else np.ones((H, W), dtype=bool)
            except Exception:
                valid_mask = np.ones((H, W), dtype=bool)
        valid_px = int(np.sum(valid_mask))
        if valid_px == 0: return 0.0
        if crs is not None and getattr(crs, "is_projected", False):
            det = abs(tr.a * tr.e - tr.b * tr.d)
            area_m2 = det * valid_px
            return area_m2 / 10000.0
        lat_c = 0.5 * (bounds.bottom + bounds.top)
        mx = 111320.0 * math.cos(math.radians(lat_c))
        my = 110540.0
        px_m2 = abs(tr.a)*mx * abs(tr.e)*my
        area_m2 = valid_px * px_m2
        return area_m2 / 10000.0

def confusion_from_masks(gt, pr, num_classes=2, valid_mask=None):
    """
    PARCHE: ahora acepta 'valid_mask' para excluir NoData.
    """
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    gt = gt.flatten(); pr = pr.flatten()
    if valid_mask is not None:
        valid = valid_mask.flatten()
    else:
        valid = np.ones_like(gt, dtype=bool)
    # además, restringe a rangos válidos de clase
    valid &= (gt >= 0) & (gt < num_classes) & (pr >= 0) & (pr < num_classes)
    gt = gt[valid]; pr = pr[valid]
    for i in range(num_classes):
        for j in range(num_classes):
            cm[i, j] += np.sum((gt == i) & (pr == j))
    return cm

def miou_pixacc_fwIoU_folder(gt_dir, pred_dir, num_classes=2):
    """
    PARCHE: usa máscara NoData del GT para el cómputo de la matriz de confusión.
    """
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    cm_total = np.zeros((num_classes, num_classes), dtype=np.int64)
    for f in files:
        gt_path = os.path.join(gt_dir, f)
        pr_path = os.path.join(pred_dir, f if os.path.exists(os.path.join(pred_dir,f)) else f.replace("Img_","Mask_"))
        if not os.path.exists(pr_path):
            continue
        gt = read_mask(gt_path); pr = read_mask(pr_path)
        with rasterio.open(gt_path) as src:
            nod = src.nodata
        valid = (gt != nod) if nod is not None else np.ones_like(gt, dtype=bool)
        cm_total += confusion_from_masks(gt, pr, num_classes=num_classes, valid_mask=valid)

    tp = np.diag(cm_total)
    total = cm_total.sum()
    pixacc = float(tp.sum()/total) if total>0 else 0.0
    ious=[]
    for c in range(num_classes):
        denom = tp[c] + (cm_total[c,:].sum()-tp[c]) + (cm_total[:,c].sum()-tp[c])
        ious.append(float(tp[c]/denom) if denom>0 else 0.0)
    miou = float(np.mean(ious)) if ious else 0.0
    freq = cm_total.sum(axis=1) / total if total>0 else np.zeros(num_classes)
    fwiou = float((freq*np.array(ious)).sum())
    return dict(mIoU=miou, pixAcc=pixacc, fwIoU=fwiou)

try:
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed
    from skimage.morphology import h_minima
    _SK_OK = True
except Exception:
    _SK_OK = False
    peak_local_max = watershed = h_minima = None
    print("scikit-image no disponible → fallback CC.")

def split_ws_pred(mask_bin, min_dist_px=3, h_rel=0.10):
    if not _SK_OK:
        labels, _ = ndimage.label(mask_bin.astype(np.uint8))
        return labels
    from scipy import ndimage as ndi
    mask_bin = mask_bin.astype(np.uint8)
    dist = ndi.distance_transform_edt(mask_bin)
    if dist.max()>0 and h_rel>0:
        try:
            dist_supp = dist - h_minima(dist, h=float(h_rel*dist.max()))
        except Exception:
            dist_supp = dist
    else:
        dist_supp = dist
    coords = peak_local_max(dist_supp, min_distance=int(max(1,min_dist_px)), labels=mask_bin)
    markers = np.zeros_like(mask_bin, dtype=np.int32)
    for i,(r,c) in enumerate(coords, start=1): markers[r,c]=i
    labels = watershed(-dist_supp, markers, mask=mask_bin)
    return labels

def coverage_density_from_folders(gt_dir, pr_dir, ws_params=None):
    """
    PARCHE:
      - Cobertura (cov_T, cov_P) calculada solo sobre píxeles válidos (NoData enmascarado).
      - Devuelve RMSE_cover/MAE_cover/MBE_cover en **%** (puntos porcentuales), no en fracción.
        *OJO*: 'series' sigue en fracción (0–1) para mantener compatibilidad con figuras.
    """
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    cov_T, cov_P, den_T, den_P = [], [], [], []
    for f in files:
        gt_path = os.path.join(gt_dir, f)
        pr_path = os.path.join(pr_dir, f if os.path.exists(os.path.join(pr_dir,f)) else f.replace("Img_","Mask_"))
        if not os.path.exists(pr_path):
            continue
        gt = read_mask(gt_path); pr = read_mask(pr_path)
        with rasterio.open(gt_path) as src:
            nod = src.nodata
        valid = (gt != nod) if nod is not None else np.ones_like(gt, dtype=bool)

        gtb = (gt==1) & valid
        prb = (pr==1) & valid

        # Cobertura en fracción (0–1) sobre píxeles válidos
        if valid.sum() == 0:
            continue
        cov_T.append(float(gtb.sum() / valid.sum()))
        cov_P.append(float(prb.sum() / valid.sum()))

        # Densidad (conteo de objetos) — con CC o WS opcional
        labs_gt,_ = ndimage.label(gtb.astype(np.uint8))
        den_T.append(int(labs_gt.max()))
        if ws_params is None:
            labs_pr,_ = ndimage.label(prb.astype(np.uint8))
            den_P.append(int(labs_pr.max()))
        else:
            labs_pr = split_ws_pred(prb.astype(np.uint8),
                                    min_dist_px=int(ws_params.get("min_dist_px",3)),
                                    h_rel=float(ws_params.get("h_rel",0.10)))
            den_P.append(int(labs_pr.max()))

    # Métricas de cobertura en % (puntos porcentuales)
    if cov_T:
        rmse_cover = float(np.sqrt(mean_squared_error(cov_T, cov_P))) * 100.0
        mae_cover  = float(np.mean(np.abs(np.array(cov_T) - np.array(cov_P)))) * 100.0
        mbe_cover  = float(np.mean(np.array(cov_P) - np.array(cov_T))) * 100.0
        r2_cover   = float(r2_score(cov_T, cov_P))
    else:
        rmse_cover = mae_cover = mbe_cover = r2_cover = float("nan")

    # Métricas de densidad (conteo por imagen) en unidades absolutas
    if den_T:
        rmse_density = float(np.sqrt(mean_squared_error(den_T, den_P)))
        mae_density  = float(np.mean(np.abs(np.array(den_T) - np.array(den_P))))
        mbe_density  = float(np.mean(np.array(den_P) - np.array(den_T)))
        r2_density   = float(r2_score(den_T, den_P))
    else:
        rmse_density = mae_density = mbe_density = r2_density = float("nan")

    out=dict(
        N=len(cov_T),
        RMSE_cover=rmse_cover, MAE_cover=mae_cover, MBE_cover=mbe_cover, R2_cover=r2_cover,
        RMSE_density=rmse_density, MAE_density=mae_density, MBE_density=mbe_density, R2_density=r2_density
    )
    return out, dict(cov_T=cov_T, cov_P=cov_P, den_T=den_T, den_P=den_P)

def eval_density_per_ha(gt_dir, pr_dir, ws_params=None):
    """
    Igual que antes, ya consideraba NoData y área válida en ha.
    """
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    yT,yP = [], []
    for f in files:
        gt_path = os.path.join(gt_dir, f)
        pr_path = os.path.join(pr_dir, f if os.path.exists(os.path.join(pr_dir,f)) else f.replace("Img_","Mask_"))
        if not os.path.exists(pr_path): continue
        gt = read_mask(gt_path); pr = read_mask(pr_path)
        with rasterio.open(gt_path) as src:
            nod = src.nodata
        valid = (gt != nod) if nod is not None else np.ones_like(gt,bool)
        area_ha = max(raster_valid_area_ha(gt_path, valid_mask=valid), 1e-9)
        gtb = (gt==1)&valid; prb = (pr==1)&valid
        labs_gt,_ = ndimage.label(gtb.astype(np.uint8)); dens_gt = int(labs_gt.max())/area_ha
        if ws_params is None:
            labs_pr,_ = ndimage.label(prb.astype(np.uint8)); dens_pr = int(labs_pr.max())/area_ha
        else:
            labs_pr = split_ws_pred(prb.astype(np.uint8), min_dist_px=int(ws_params.get("min_dist_px",3)), h_rel=float(ws_params.get("h_rel",0.10)))
            dens_pr = int(labs_pr.max())/area_ha
        yT.append(dens_gt); yP.append(dens_pr)
    rmse=float(np.sqrt(mean_squared_error(yT,yP))) if yT else np.nan
    r2=float(r2_score(yT,yP)) if yT else np.nan
    return dict(y_true=yT, y_pred=yP, rmse=rmse, r2=r2)

# === Re-evaluación con las funciones parcheadas ===
pix_PI = miou_pixacc_fwIoU_folder(GT_PI, PRED_PI_TH, num_classes=2)
pix_FW = miou_pixacc_fwIoU_folder(GT_FW, PRED_FW_TH, num_classes=2)
print("Pixel PI:", pix_PI); print("Pixel FW:", pix_FW)

WS_PARAMS = dict(min_dist_px=3, h_rel=0.10)
pi_cd, _ = coverage_density_from_folders(GT_PI, PRED_PI_TH, ws_params=WS_PARAMS)
fw_cd, _ = coverage_density_from_folders(GT_FW, PRED_FW_TH, ws_params=WS_PARAMS)
pi_ha = eval_density_per_ha(GT_PI, PRED_PI_TH, ws_params=WS_PARAMS)
fw_ha = eval_density_per_ha(GT_FW, PRED_FW_TH, ws_params=WS_PARAMS)

print("\nCobertura/Densidad baseline (θ* | WS baseline):")
print("PI: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    pi_cd["RMSE_cover"], pi_cd["MAE_cover"], pi_cd["MBE_cover"], pi_cd["R2_cover"],
    pi_cd["RMSE_density"], pi_cd["MAE_density"], pi_cd["MBE_density"], pi_cd["R2_density"]))
print("FW: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    fw_cd["RMSE_cover"], fw_cd["MAE_cover"], fw_cd["MBE_cover"], fw_cd["R2_cover"],
    fw_cd["RMSE_density"], fw_cd["MAE_density"], fw_cd["MBE_density"], fw_cd["R2_density"]))

print("\nDensidad por hectárea baseline (θ* | WS baseline):")
print("PI: RMSE={:.2f} ind/ha | R²={:.3f}".format(pi_ha["rmse"], pi_ha["r2"]))
print("FW: RMSE={:.2f} ind/ha | R²={:.3f}".format(fw_ha["rmse"], fw_ha["r2"]))


def calibrate_ws_density(gt_dir, pr_dir, grid_min_dist=(2,3,4,5,6,7), grid_h=(0.05,0.08,0.10,0.12,0.15,0.20)):
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    best=dict(rmse=1e9, min_dist_px=None, h_rel=None)
    for dmin in grid_min_dist:
        for h in grid_h:
            yT,yP=[],[]
            for f in files:
                gt_path = os.path.join(gt_dir, f)
                pr_path = os.path.join(pr_dir, f if os.path.exists(os.path.join(pr_dir,f)) else f.replace("Img_","Mask_"))
                if not os.path.exists(pr_path): continue
                gt = read_mask(gt_path); pr = read_mask(pr_path)
                gtb = (gt==1); prb=(pr==1)
                labs_gt,_ = ndimage.label(gtb.astype(np.uint8)); dens_gt=int(labs_gt.max())
                labs_pr = split_ws_pred(prb.astype(np.uint8), min_dist_px=int(dmin), h_rel=float(h))
                dens_pr=int(labs_pr.max())
                yT.append(dens_gt); yP.append(dens_pr)
            if yT:
                rmse=float(np.sqrt(mean_squared_error(yT,yP)))
                if rmse < best["rmse"]: best=dict(rmse=rmse, min_dist_px=int(dmin), h_rel=float(h))
    return best

best_ws_fw = calibrate_ws_density(GT_FW, PRED_FW_TH)
print("\nWS FW óptimo:", best_ws_fw)

WS_PARAMS_FW = dict(min_dist_px=int(best_ws_fw["min_dist_px"]), h_rel=float(best_ws_fw["h_rel"]))
fw_cd_cal, _ = coverage_density_from_folders(GT_FW, PRED_FW_TH, ws_params=WS_PARAMS_FW)
fw_ha_cal = eval_density_per_ha(GT_FW, PRED_FW_TH, ws_params=WS_PARAMS_FW)

print("\nCobertura/Densidad FW (θ* + WS calibrado):")
print("FW: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    fw_cd_cal["RMSE_cover"], fw_cd_cal["MAE_cover"], fw_cd_cal["MBE_cover"], fw_cd_cal["R2_cover"],
    fw_cd_cal["RMSE_density"], fw_cd_cal["MAE_density"], fw_cd_cal["MBE_density"], fw_cd_cal["R2_density"]))

print("\nDensidad por hectárea FW (θ* + WS calibrado):")
print("FW: RMSE={:.2f} ind/ha | R²={:.3f}".format(fw_ha_cal["rmse"], fw_ha_cal["r2"]))

df_covdens = pd.DataFrame([
    dict(Split="PI-test", **pi_cd, RMSE_dens_ha=pi_ha["rmse"], R2_dens_ha=pi_ha["r2"]),
    dict(Split="FW-test (WS base)", **fw_cd, RMSE_dens_ha=fw_ha["rmse"], R2_dens_ha=fw_ha["r2"]),
    dict(Split="FW-test (WS cal)", **fw_cd_cal, RMSE_dens_ha=fw_ha_cal["rmse"], R2_dens_ha=fw_ha_cal["r2"]),
])

# Guardar resultados en CSV y LaTeX
out_csv = f"{OUT_PI}/csv/coverage_density_MaskRCNN_theta_star.csv"
out_tex = f"{OUT_PI}/tables/table_coverage_density_MaskRCNN_theta_star.tex"
df_covdens.to_csv(out_csv, index=False)
with open(out_tex, "w") as f:
    f.write(df_covdens.to_latex(index=False, float_format="%.4f"))

print("\nCobertura/densidad guardadas (baseline y WS calibrado).")
print(f"CSV: {out_csv}")
print(f"LaTeX: {out_tex}")

In [ ]:
#@title 9. Figuras extra: Scatter (FW) y Barras por tamaño (PI/FW)

import matplotlib.pyplot as plt
from scipy.stats import pearsonr

def _rmse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2))) if len(y_true) else float("nan")
def _mae(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(np.abs(y_true - y_pred))) if len(y_true) else float("nan")
def _mbe(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(y_pred - y_true)) if len(y_true) else float("nan")

os.makedirs(f"{OUT_PI}/plots", exist_ok=True)

try:
    _WS = WS_PARAMS_FW
except NameError:
    try:
        _WS = WS_PARAMS
    except NameError:
        _WS = dict(min_dist_px=3, h_rel=0.10)

_, _series_fw = coverage_density_from_folders(GT_FW, PRED_FW_TH, ws_params=_WS)
_cov_T = np.array(_series_fw["cov_T"]) * 100.0
_cov_P = np.array(_series_fw["cov_P"]) * 100.0

_den_fw = eval_density_per_ha(GT_FW, PRED_FW_TH, ws_params=_WS)
_den_T = np.array(_den_fw["y_true"])
_den_P = np.array(_den_fw["y_pred"])

_theta_fw_txt = f"{theta_fw:.2f}" if 'theta_fw' in globals() else "?"

plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "legend.fontsize": 10, "figure.dpi": 150
})
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

r_cov = pearsonr(_cov_T, _cov_P)[0] if len(_cov_T) > 1 else np.nan
rmse_cov, mae_cov, mbe_cov = _rmse(_cov_T, _cov_P), _mae(_cov_T, _cov_P), _mbe(_cov_T, _cov_P)
axs[0].scatter(_cov_T, _cov_P, s=12)
lim = [0, max(1e-6, _cov_T.max(), _cov_P.max()) * 1.05]
axs[0].plot(lim, lim, linestyle="--")
axs[0].set_xlim(lim); axs[0].set_ylim(lim)
axs[0].set_xlabel("Observed canopy cover (%)"); axs[0].set_ylabel("Predicted canopy cover (%)")
axs[0].set_title(f"(a) FW — Cover @ θ={_theta_fw_txt}")
axs[0].text(0.02, 0.98, f"r={r_cov:.2f}\nRMSE={rmse_cov:.2f}%\nMAE={mae_cov:.2f}%\nMBE={mbe_cov:.2f}%",
            transform=axs[0].transAxes, va="top")

r_den = pearsonr(_den_T, _den_P)[0] if len(_den_T) > 1 else np.nan
rmse_den, mae_den, mbe_den = _rmse(_den_T, _den_P), _mae(_den_T, _den_P), _mbe(_den_T, _den_P)
axs[1].scatter(_den_T, _den_P, s=12)
lim2 = [0, max(1e-6, _den_T.max(), _den_P.max()) * 1.05]
axs[1].plot(lim2, lim2, linestyle="--")
axs[1].set_xlim(lim2); axs[1].set_ylim(lim2)
axs[1].set_xlabel("Observed shrubs per ha"); axs[1].set_ylabel("Predicted shrubs per ha")
axs[1].set_title(f"(b) FW — Density @ θ={_theta_fw_txt}")
axs[1].text(0.02, 0.98, f"r={r_den:.2f}\nRMSE={rmse_den:.2f}\nMAE={mae_den:.2f}\nMBE={mbe_den:.2f}",
            transform=axs[1].transAxes, va="top")

fig.suptitle("Observed vs Predicted — FW — Mask R-CNN")
fig.tight_layout(rect=[0, 0, 1, 0.93])
_scatter_path = f"{OUT_PI}/plots/FW_scatter_cover_density.png"
plt.savefig(_scatter_path, bbox_inches="tight")
plt.close()

print("Scatter FW guardado en:", _scatter_path)

_theta_pi = globals().get('theta_pi', 0.5)
_theta_fw = globals().get('theta_fw', 0.5)

def _get_sizewise(coco, ims, img_dir, predictor, theta):
    return sizewise_metrics(coco, ims, img_dir, predictor, theta)

try:
    _df_pi_sz = df_pi_sz.copy()
except NameError:
    _df_pi_sz = _get_sizewise(coco_pi, ims_pi, PI_IMG_DIR, predictor, _theta_pi)

try:
    _df_fw_sz = df_fw_sz.copy()
except NameError:
    _df_fw_sz = _get_sizewise(coco_fw, ims_fw, FW_IMG_DIR, predictor, _theta_fw)

_order = [b[0] for b in SIZE_BINS] + ["All"]
_df_pi_sz = _df_pi_sz.set_index("Size").reindex(_order).reset_index()
_df_fw_sz = _df_fw_sz.set_index("Size").reindex(_order).reset_index()

def _plot_sizewise_bars(df_sz, split_name, out_name, theta_txt):
    if not {"IoU_F1","S-IoU_F1","Size"}.issubset(df_sz.columns):
        raise RuntimeError("El DataFrame de tamaños no contiene columnas esperadas: IoU_F1, S-IoU_F1, Size")
    labels = df_sz["Size"].tolist()
    x = np.arange(len(labels))
    width = 0.38

    plt.rcParams.update({
        "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
        "legend.fontsize": 10, "figure.dpi": 150
    })
    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    ax.bar(x - width/2, df_sz["IoU_F1"].values, width, label="IoU @ 0.5")
    ax.bar(x + width/2, df_sz["S-IoU_F1"].values, width, label="S-IoU @ 0.5")
    ax.set_xticks(x, labels)
    ax.set_ylabel("F1-score (%)")
    ax.set_xlabel("Size bin")
    ax.set_title(f"{split_name} — Size-wise F1 (θ={theta_txt}) — Mask R-CNN")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="best", frameon=False)
    _out = f"{OUT_PI}/plots/{out_name}"
    plt.savefig(_out, bbox_inches="tight")
    plt.close()
    return _out

_pi_bars_path = _plot_sizewise_bars(_df_pi_sz, "PI", "PI_sizewise_bars.png", f"{_theta_pi:.2f}")
_fw_bars_path = _plot_sizewise_bars(_df_fw_sz, "FW", "FW_sizewise_bars.png", f"{_theta_fw:.2f}")

print("Barras por tamaño guardadas en:")
print("-", _pi_bars_path)
print("-", _fw_bars_path)